# 01 - Online Retail II: Data Preparation

**Dataset**: Online Retail II (UK-based online retailer, giftware, mostly wholesale)
**Source**: UCI Machine Learning Repository
**Transactions**: 1,067,371 lines | **Period**: Dec. 2009 - Dec. 2011 | **Variables**: 9
**Currency**: GBP (£)

---
## Business Context & Objectives

This notebook loads the **Online Retail II** dataset and establishes the foundational data cleaning decisions that all downstream analyses (customer segmentation in Notebook 02, retention/churn in 03, repurchase prediction in 04) depend on.

### Key Objectives
1. **Raw observation:** explore the raw data (schema, types, missingness, volume) without applying any filter yet.
2. **Data integrity & edge cases:** identify and handle retail-specific anomalies (invoice cancellations, non-product stock codes, zero/negative prices, system test data).
3. **Traceability:** quantify the impact of each cleaning decision in both rows removed and revenue affected.
4. **Decision logging:** document every assumption so downstream notebooks inherit an auditable, not a silent, set of choices.

## Research Questions

1. **Data quality**: what does a transaction line actually represent, and which lines are unusable for customer-level analysis?
2. **Scope**: once cleaned, what customer base and what share of revenue remains for segmentation, retention and prediction (notebooks 02-04)?

---
### Performance note
The raw `.xlsx` (~45 MB) is read once and cached locally as Parquet. Re-running the notebook then takes seconds instead of minutes.

---
## Key Variables

| Type | Variables |
|---|---|
| **Transaction** | `Invoice`, `InvoiceDate`, `Quantity`, `Price` |
| **Product** | `StockCode`, `Description` |
| **Customer** | `Customer ID`, `Country` |
| **Derived** | `IsCancellation`, `IsNonProduct`, `LineRevenue` |

---
## Plan

0. Setup & Loading
1. General Overview
2. Data Quality & Retail Edge Cases
3. Data Cleaning & Transformation Pipeline
4. Summary & Decision Log


# 0. Setup & Loading

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

RAW = Path("../data/raw/online_retail_II.xlsx")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)
PARQUET_PATH = PROC / "raw_concat.parquet"

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    source_info = f"Loaded dataset from Parquet cache (`{PARQUET_PATH.name}`)"
else:
    sheets = pd.read_excel(RAW, sheet_name=None)
    df = pd.concat(
        [sheet.assign(SourceSheet=name) for name, sheet in sheets.items()],
        ignore_index=True,
    )
    df.columns = df.columns.str.strip()

    # 'Invoice' mixes integers and cancellation codes ('C489449');
    # 'StockCode' mixes numeric codes and non-product codes (POST, DOT, M...);
    # 'Description' mixes a few numeric values among free text.
    # 'string' dtype preserves missing values as <NA> instead of literal "nan".
    text_cols = df.select_dtypes(include="object").columns
    df[text_cols] = df[text_cols].apply(lambda s: s.astype("string").str.strip())

    df.to_parquet(PARQUET_PATH, index=False)
    source_info = f"Parsed raw Excel sheets and cached to `{PARQUET_PATH.name}`"

display(Markdown(f"### Data Loading Completed\n* **Source:** {source_info}"))


# 1. General Overview

## 1.1 Dataset Summary & Schema

In [ ]:
sheets_list = ", ".join(df["SourceSheet"].unique())
min_date = df["InvoiceDate"].min()
max_date = df["InvoiceDate"].max()

display(Markdown(f"""
### Dataset Summary
* **Dimensions:** `{df.shape[0]:,}` rows x `{df.shape[1]}` columns
* **Source sheets:** {sheets_list}
* **Date range:** from `{min_date}` to `{max_date}`
"""))
display(df.head())


## 1.2 Proactive Type Checking and Identifier Inspection

Before casting types or performing calculations, we check that identifiers such as `Invoice` or `StockCode` are homogeneous.

In [ ]:
display(Markdown("### Column Types"))
dtypes_df = pd.DataFrame(df.dtypes, columns=["Data Type"]).reset_index()
dtypes_df.columns = ["Column", "Type"]
display(dtypes_df)


In [ ]:
non_num_invoices = df[df["Invoice"].str.contains(r"[A-Za-z]", na=False)]
non_num_stockcodes = df[df["StockCode"].str.contains(r"[A-Za-z]", na=False)]

display(Markdown(f"* **Invoices with letters:** `{len(non_num_invoices):,}` ({len(non_num_invoices)/len(df):.2%})"))
display(Markdown(f"* **StockCodes with letters:** `{len(non_num_stockcodes):,}` ({len(non_num_stockcodes)/len(df):.2%})"))
display(Markdown("### Non-numeric characters in Invoice"))
display(non_num_invoices.head())
display(Markdown("### Non-numeric characters in StockCode"))
display(non_num_stockcodes.head())


**Observation:** non-numeric invoices start with `C` (cancellations, negative quantity) — and, as discovered later in section 2, a small number start with `A` (accounting adjustments, e.g. `StockCode = "B"`). Both are isolated as distinct signals rather than silently netted; the netting decision is made explicitly in notebook 02, not here.

# 1.3 Missing Values & Uniqueness Audit

In [ ]:
line_revenue = (df["Quantity"] * df["Price"]).fillna(0.0)
total_revenue = line_revenue.sum()

n_missing = df.isna().sum()
pct_missing = (n_missing / len(df) * 100).round(2)
revenue_missing = df.isna().T.dot(line_revenue)
pct_revenue_missing = (revenue_missing / total_revenue * 100).fillna(0).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": n_missing,
    "Percentage (%)": pct_missing,
    "Revenue Missing (GBP)": revenue_missing,
    "Revenue Missing (%)": pct_revenue_missing,
}).sort_values(by="Missing Values", ascending=False)

display(Markdown("### Missing Values & Financial Impact Breakdown"))
display(missing_summary.style.format({
    "Missing Values": "{:,}",
    "Percentage (%)": "{:.2f}%",
    "Revenue Missing (GBP)": "£{:,.2f}",
    "Revenue Missing (%)": "{:.2f}%",
}))

display(Markdown(f"""
### Unique Entities
* **Unique customers:** `{df['Customer ID'].nunique():,}`
* **Unique invoices:** `{df['Invoice'].nunique():,}`
* **Unique stock codes:** `{df['StockCode'].nunique():,}`
"""))


## First observations

- **1,067,371 transaction lines** across two trading years (Dec. 2009 - Dec. 2011), from a UK-based online retailer selling giftware, mostly to wholesale buyers.
- **`Customer ID` is missing on 22.77% of lines** (guest checkouts or non-attributed sales) — unusable for customer-level analysis, excluded in section 3.
- **Missing-ID rows are lower value on average**: 22.77% of rows but only 13.68% of revenue.
- **`Description` is missing on 0.41% of lines** — negligible.


# 2. Data Quality & Retail Edge Cases

## 2.1 Cancellation Analysis (`C` Invoice Prefix)

In [ ]:
cancellations = df[df["Invoice"].str.startswith("C", na=False)]
cancellation_revenue = (cancellations["Quantity"] * cancellations["Price"]).sum()

display(Markdown(f"""
### Cancellation Summary (`C` prefix)
* **Total cancellation rows:** `{len(cancellations):,}` ({len(cancellations)/len(df):.2%})
* **Total negative quantity:** `{cancellations['Quantity'].sum():,}`
* **Net financial impact:** `£{cancellation_revenue:,.2f}`
"""))


**Anomaly check.** One cancellation row (`C496350`, `StockCode = "M"` — manual entry) carries a positive quantity. It has no `Customer ID` attached and is excluded from customer-level analysis regardless (section 3).

**Conclusion — cancellations.** They represent 1.83% of rows but -7.9% of total signed revenue. They are flagged (`IsCancellation`), not silently netted, so notebook 02 decides explicitly how to treat them.

## 2.2 Price and Quantity Anomalies

In [ ]:
neg_qty_no_c = df[(df["Quantity"] < 0) & (~df["Invoice"].str.startswith("C", na=False))]
pos_qty_with_c = df[(df["Quantity"] > 0) & (df["Invoice"].str.startswith("C", na=False))]
zero_or_neg_price = df[df["Price"] <= 0]

display(Markdown(f"""
### Data Integrity Anomalies
* **Negative quantity without 'C':** `{len(neg_qty_no_c):,}` rows
* **Cancellation ('C') with positive quantity:** `{len(pos_qty_with_c):,}` rows
* **Zero or negative price:** `{len(zero_or_neg_price):,}` rows
"""))


**Investigation.** Each anomaly was checked against real rows rather than assumed:

- **`neg_qty_no_c` (3,457 rows): 100% have no `Customer ID`.** Fully absorbed by the missing-ID exclusion in section 3 — no separate rule needed.
- **`zero_or_neg_price` (6,207 rows): the large negative values (5 rows, e.g. -£53,594.36) are all `StockCode = "B"`**, `Description = "Adjust bad debt"` — an accounting write-off, confirmed via description text, not guessed. **98.86% of the remaining zero-price rows have no `Customer ID`**, also absorbed by section 3.
- **The 71 zero-price rows with a `Customer ID`** are mostly genuine products (real descriptions, plausible quantities) given at zero cost — promotional gifts or goodwill gestures to real customers. They are **kept as-is**: removing them would misrepresent what actually happened.
- Within that residual, **`StockCode TEST001` and `TEST002`** were found: `Description = "This is a test product."`, 17 rows total across the full dataset, at various prices (not only zero) — meaning the alpha-only filter in section 2.3 could never have caught them on its own. Customer `12346`, flagged by these test rows, was checked individually and confirmed to be a genuine customer with real purchases across several months (parasols, doormats); only the two test-code rows are excluded, not the customer.

**Note on method.** The alpha-only regex filter (`^[A-Za-z]+$`) used in section 2.3 is blind to alphanumeric codes like `TEST001`, and to descriptions rather than codes. `TEST001`, `TEST002` and `B` only surfaced by cross-checking these price/quantity anomalies against real row samples — not by trusting aggregate counts. Every anomaly in this notebook is resolved by inspecting actual rows before writing a decision.

## 2.3 Non-Product Codes Analysis

`StockCode` may mix genuine products with codes representing administrative or non-merchandise entries. Checked directly below rather than assumed.

In [ ]:
non_product_codes = df.loc[
    df["StockCode"].str.match(r"^[A-Za-z]+$", na=False),
    "StockCode"
].value_counts()

display(Markdown("### Non-Product Codes Overview (alpha-only codes)"))
display(Markdown(f"*Found **`{len(non_product_codes)}`** unique alphabetic stock codes.*"))
npc_df = non_product_codes.reset_index()
npc_df.columns = ["StockCode", "Occurrence Count"]
display(npc_df)


The alpha-only filter over-catches genuine products (`DCGSSGIRL`, `DCGSSBOY`, `DCGSLGIRL`, `DCGSLBOY` — a childrenswear line — and `PADS`), which are **kept**. `GIFT` is also kept as a sellable item.

It also **misses** `TEST001`, `TEST002` (alphanumeric) — found only via section 2.2's price/quantity investigation.

Confirmed non-product / administrative codes, each verified against its `Description` or transaction pattern:

| Code | Description | Evidence |
|---|---|---|
| `POST` | Postage | Value counts, plausible volume |
| `DOT` | Dotcom postage / charge | Value counts |
| `M` / `m` | Manual entry | Value counts, merged case-insensitive |
| `D` | Discount | Value counts |
| `S` | Samples | Value counts |
| `ADJUST` | Accounting adjustment | `Description` e.g. "Adjustment by john on..." |
| `AMAZONFEE` | Amazon platform fee | Value counts, large negative average |
| `CRUK` | Cancer Research UK donation | Value counts |
| `B` | Bad debt adjustment | `Description = "Adjust bad debt"` |
| `TEST001`, `TEST002` | System test data | `Description = "This is a test product."` |


In [ ]:
NON_PRODUCT_CODES = {
    "POST": "Postage",
    "DOT": "Dotcom postage / charge",
    "M": "Manual entry",
    "D": "Discount",
    "S": "Samples",
    "ADJUST": "Accounting adjustment",
    "AMAZONFEE": "Amazon platform fee",
    "CRUK": "Cancer Research UK donation",
    "B": "Bad debt adjustment",
    "TEST001": "Test product (system test data)",
    "TEST002": "Test product (system test data)",
}

df["StockCodeUpper"] = df["StockCode"].str.upper()
df["IsNonProduct"] = df["StockCodeUpper"].isin(NON_PRODUCT_CODES.keys())

non_product_df = df[df["IsNonProduct"]]
non_product_summary = (
    non_product_df.groupby("StockCodeUpper")
    .apply(
        lambda g: pd.Series({
            "Description": NON_PRODUCT_CODES.get(g.name, "Other"),
            "Total Rows": len(g),
            "Total Quantity": g["Quantity"].sum(),
            "Total Revenue (GBP)": (g["Quantity"] * g["Price"]).sum(),
        }),
        include_groups=False,
    )
    .reset_index()
)

display(Markdown("### Non-Product Codes Overview (final)"))
display(non_product_summary.style.format({
    "Total Rows": "{:,}",
    "Total Quantity": "{:,}",
    "Total Revenue (GBP)": "£{:,.2f}",
}))


These lines carry no product or customer-behaviour information and are excluded from the analytical base used in notebooks 02-04. They are kept in the raw table (flagged via `IsNonProduct`, not deleted) so the exclusion remains auditable.

# 3. Data Cleaning & Transformation Pipeline

Two exclusions are applied together to build the customer-level analytical base:

1. **Non-product lines** (`IsNonProduct`) excluded — including the `B`, `TEST001`, `TEST002` codes confirmed in section 2.
2. **Rows without `Customer ID`** excluded.

**Cancellations are kept, not excluded here.** Whether to net them into a customer's monetary value or treat cancellation behaviour as a separate signal is decided explicitly in notebook 02.

## 3.1 Filtering & Feature Engineering

In [ ]:
initial_rows = len(df)
total_raw_revenue = (df["Quantity"] * df["Price"]).sum()

if "StockCodeUpper" not in df.columns:
    df["StockCodeUpper"] = df["StockCode"].str.upper()
if "IsNonProduct" not in df.columns:
    df["IsNonProduct"] = df["StockCodeUpper"].isin(NON_PRODUCT_CODES.keys())
df["IsCancellation"] = df["Invoice"].str.startswith("C", na=False)

step1 = df.loc[~df["IsNonProduct"]]
rows_after_non_product = len(step1)

df_clean = step1.loc[step1["Customer ID"].notna()].copy()
rows_final = len(df_clean)

df_clean["Customer ID"] = df_clean["Customer ID"].astype("int64").astype("string")
df_clean["LineRevenue"] = df_clean["Quantity"] * df_clean["Price"]

pct_clean = (rows_final / initial_rows) * 100
n_unique_cust = df_clean["Customer ID"].nunique()
revenue_retained = df_clean["LineRevenue"].sum()
pct_revenue_retained = (revenue_retained / total_raw_revenue) * 100
n_cancellations_kept = df_clean["IsCancellation"].sum()

display(Markdown(f"""
### Cleaned Dataset Ready
* **Analytical base:** `{rows_final:,}` rows (`{pct_clean:.2f}%` of raw data)
* **Unique customers:** `{n_unique_cust:,}`
* **Revenue retained:** `£{revenue_retained:,.2f}` (`{pct_revenue_retained:.2f}%` of raw total revenue)
* **Cancellation rows kept (flagged):** `{n_cancellations_kept:,}`
"""))


**To verify after this update:** these figures should be very close to the previous run (820,963 rows / 5,882 customers / 86.73% revenue), since `B`, `TEST001` and `TEST002` together represent a handful of rows. Re-run and confirm before moving to notebook 02 — do not assume the numbers are unchanged.

## 3.2 Clean Dataset Export

In [ ]:
out_path = PROC / "clean_transactions.parquet"
df_clean.to_parquet(out_path, index=False)
display(Markdown(f"Saved successfully to: `{out_path}`"))


# 4. Summary & Decision Log

## 4.1 Impact Quantification

In [ ]:
decision_log = pd.DataFrame([
    {"Step": "Raw data", "Condition": "Initial dataset",
     "Rows Remaining": initial_rows, "Rows Removed": 0, "% Retained": "100.0%"},
    {"Step": "1. Non-product codes",
     "Condition": "Remove POST, DOT, M, D, S, ADJUST, AMAZONFEE, CRUK, B, TEST001, TEST002",
     "Rows Remaining": rows_after_non_product,
     "Rows Removed": initial_rows - rows_after_non_product,
     "% Retained": f"{rows_after_non_product/initial_rows:.1%}"},
    {"Step": "2. Missing Customer ID", "Condition": "Remove rows without a Customer ID",
     "Rows Remaining": rows_final, "Rows Removed": rows_after_non_product - rows_final,
     "% Retained": f"{rows_final/initial_rows:.1%}"},
])
display(Markdown("### Cleaning Decision Log & Impact Summary"))
display(decision_log)


## Decision log summary

| # | Decision | Notes |
|---|----------|-------|
| 1 | Non-product codes excluded | `POST`, `DOT`, `M`, `D`, `S`, `ADJUST`, `AMAZONFEE`, `CRUK`, `B`, `TEST001`, `TEST002` — each verified via `Description` or value counts |
| 2 | Missing Customer ID excluded | Absorbs 100% of the negative-quantity and 98.86% of the zero-price anomalies from section 2.2 |
| 3 | Cancellations flagged, **kept** (not netted) | Netting decision deferred to notebook 02 |
| 4 | 71 zero-price rows with a real customer | Kept — genuine promotional gifts, not anomalies |

This table (`clean_transactions.parquet`) is the single source for notebooks 02-04.

## Summary

The analytical base retains the large majority of raw rows and revenue, across the identified customer base spanning Dec. 2009 - Dec. 2011, with cancellations preserved and flagged rather than removed. Every exclusion in this notebook was resolved by inspecting real rows, not by acting on aggregate counts alone — three codes (`B`, `TEST001`, `TEST002`) would have been missed entirely by the alpha-only filter used in section 2.3.
